# Agentic AI vs. AI Agents

### A one-stop introduction — from a single LLM call to a multi-agent system

This notebook answers one question properly: **is there a difference between "AI agents" and "Agentic AI", or is it just marketing?**

The short answer is that there *is* a real difference, but it is not the difference most people assume. Rather than argue about definitions in the abstract, we are going to **build** our way up the ladder — starting from a plain LLM call that has no agency at all, and adding one capability at a time until we have a multi-agent system. By the end you will be able to point at a specific line of code and say "*this* is the line that turned a program into an agent."

**What you will build**

| # | We build | Concept it teaches |
|---|---|---|
| 0 | A plain LLM call | The baseline: no agency |
| 1 | A chain | Fixed control flow (a *workflow*) |
| 2 | A router | The model's first decision |
| 3 | Tool calling | The model chooses an *action* |
| 4 | An agent loop, from scratch | **This is where an AI agent is born** |
| 5 | The same thing with `create_agent` | The production shortcut |
| 6 | Memory across turns | Statefulness |
| 7 | Reflection & planning loops | Agentic design patterns |
| 8 | A supervisor + specialists | **This is where "Agentic AI" lives** |
| 9 | Limits & human approval | Control and safety |

**Setup**: everything runs on the free [Groq](https://console.groq.com) API using `qwen/qwen3.6-27b`, the same model as the rest of this repo. You need a `.env` file in this folder containing `GROQ_API_KEY=...` (you already have one). Run the cells top to bottom.

---
## Part 1 — The terminology, settled

### The short answer

- **"AI agent"** is a **noun**. It names a *thing you can build, run and count*. You can have three agents.
- **"Agentic"** is an **adjective**. It describes a *property* — how much independent decision-making a system has. A system can be *slightly* agentic or *very* agentic.
- **"Agentic AI"** is the **paradigm/field** built on that property: the whole design approach of giving AI systems goals instead of instructions, plus (increasingly) the specific label for **orchestrated, multi-agent, higher-autonomy systems** as opposed to a single tool-calling bot.

So they are not synonyms, but they are also not rivals. The cleanest way to say it:

> **An AI agent is the unit. Agentic AI is the property that unit has, and the discipline of designing systems around it.**

An analogy that holds up well: *a car* is to *mobility* what *an AI agent* is to *Agentic AI*. One is a countable artifact; the other is the capability and the field of engineering around it.

### The precise answer, side by side

| | **AI Agent** | **Agentic AI** |
|---|---|---|
| **Part of speech** | Noun — a concrete artifact | Adjective + field — a property and a paradigm |
| **Scope** | One system: model + tools + loop | The design approach; often **many** agents cooperating |
| **You would say** | "I deployed *an agent* that triages tickets" | "We're moving our support stack to *agentic AI*" |
| **Typical structure** | A single loop calling tools until done | Supervisor + specialists, planners, critics, shared memory |
| **Autonomy** | Bounded — one goal, one toolset | Higher — decomposes goals, delegates, self-corrects |
| **Failure mode** | Wrong tool, wrong argument, infinite loop | Coordination failure, error propagation between agents, runaway cost |
| **Analogy** | An employee | The org chart and the working culture |

### The honest caveat

Nobody owns these terms. Vendors use "Agentic AI" for things that are a single `if` statement, and researchers sometimes use "agent" for an entire multi-agent swarm. There is **no standards body** here. What matters is not policing the words but being able to answer the real question underneath them:

> **How much of the control flow is decided by my code, and how much is decided by the model at runtime?**

That question has a precise answer for any system you build, and the rest of this notebook is about learning to answer it.

### The distinction that actually matters: workflow vs. agent

A widely used and much more rigorous framing (popularised by Anthropic's *Building Effective Agents*) cuts the space in two:

- **Workflow** — the LLM is a component inside **control flow you wrote**. The path through the code is fixed at design time. You know in advance which steps run and in what order.
- **Agent** — the LLM **directs its own control flow** at runtime. It decides which tools to call, in what order, and when the job is finished. You do *not* know the path in advance.

Everything else — "autonomous", "goal-driven", "reasoning" — follows from that one distinction. Levels 0–2 below are workflows. Levels 4+ are agents. Level 3 is the hinge.

### The autonomy ladder

```
                                 WHO DECIDES WHAT HAPPENS NEXT?
                                 ================================

 L0  Plain LLM call        [you]------------------------------------------  no agency
     prompt -> text                                                          a function call

 L1  Chain                 [you]------------------------------------------  no agency
     prompt -> LLM -> prompt -> LLM -> out                                   fixed pipeline

 L2  Router                [you]--------------------------[model]---------   a decision
     LLM picks a branch, your code runs it                                   1 choice, then done

 L3  Tool calling          [you]-----------------[model]------------------   an intent
     LLM says "call f(x)" -- but nothing runs it yet                         still one shot
     ------------------------------- THE LINE -------------------------------
 L4  Agent (the loop)      [you]------[model]---------------------------->   AN AI AGENT
     LLM acts, sees results, decides again, until done                       runtime control flow

 L5  Multi-agent system    [you]--[model]-------------------------------->   AGENTIC AI
     agents delegate to agents, plan, critique, retry                        emergent trajectories
```

**The line between L3 and L4 is the entire subject of this notebook.** Below it, the model *advises*. Above it, the model *drives*. Let's go build both sides of that line.

---
## Part 2 — Setup

We use `init_chat_model`, which gives a provider-neutral handle on the model. The `"groq:"` prefix tells LangChain which provider to route to — swapping to `"google_genai:gemini-2.5-flash"` later would change nothing else in this notebook.

Two parameters matter a lot here, because `qwen3.6` is a **reasoning model** — it thinks at length before answering:

- **`reasoning_format="parsed"`** tells Groq to put that thinking in a separate field instead of inline in the answer. Without it, every `content` starts with a `<think>...</think>` block and the notebook becomes unreadable.
- **`max_tokens=4096`** gives it room. Reasoning models genuinely need it: with the default cap, a long thinking block can consume the entire budget and the response gets truncated *before the actual answer is written*.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()                                    # reads .env from this folder
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

MODEL_ID = "groq:qwen/qwen3.6-27b"

model = init_chat_model(
    MODEL_ID,
    reasoning_format="parsed",   # keep <think> out of .content
    max_tokens=4096,             # leave room for thinking AND the answer
)

print("Model ready:", MODEL_ID)

### Where did the thinking go?

It is still there — just moved out of the way, into `additional_kwargs["reasoning_content"]`. This is worth seeing once, because the model's scratchpad is often the fastest way to understand *why* an agent did something strange.

In [ ]:
probe = model.invoke("What is 17 * 23? Answer with just the number.")

print("ANSWER   :", probe.content)
print("\nTHINKING (first 400 chars):\n", probe.additional_kwargs["reasoning_content"][:400])

### Two small helpers

`clean()` is a defensive backstop — with `reasoning_format="parsed"` the content is already clean, but if you ever swap models or drop that setting, stray `<think>` tags reappear. `show()` just wraps and truncates so long answers stay readable.

Note `msg.text` — that is the provider-neutral way to get a message's text. `msg.content` also works here, but `.text` flattens multimodal content blocks into a plain string on every provider, so it is the safer habit.

In [ ]:
import re
import textwrap

def clean(msg_or_text) -> str:
    """Return plain text, stripping a <think> block if one leaked through."""
    text = msg_or_text.text if hasattr(msg_or_text, "text") else msg_or_text
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)   # unterminated (truncated) block
    return text.strip()

def show(msg_or_text, width=100, limit=1200):
    """Pretty-print a model response, wrapped and truncated."""
    text = clean(msg_or_text)
    if len(text) > limit:
        text = text[:limit] + f"\n... [truncated, {len(text)} chars total]"
    print("\n".join(textwrap.fill(line, width) for line in text.split("\n")))

print("Helpers loaded.")

---
## Level 0 — A plain LLM call

### There is no agent here at all

This is the baseline. Text goes in, text comes out. The model cannot check a fact, cannot run code, cannot look anything up, and cannot take a second attempt. It is a **pure function** from a string to a string.

In [ ]:
response = model.invoke("Name three classical Indian musical instruments.")

show(response)

print("\n" + "=" * 70)
print("tool_calls    :", response.tool_calls)          # empty -- it asked for nothing
print("finish_reason :", response.response_metadata["finish_reason"])

### What to notice

- `tool_calls` is `[]` and `finish_reason` is `'stop'`. The model produced words and stopped. That is the signature of a non-agentic response.
- **The model took no action in the world.** Nothing outside the model changed.
- If the answer were wrong, nothing would catch it. There is no feedback of any kind.

Autonomy score: **zero**. Every decision about what happens next was made by you when you typed `.invoke()`.

---
## Level 1 — A chain

### More steps, still zero agency

Now we run two LLM calls in sequence: summarise a text, then translate the summary. This is a **chain**, built with LangChain's pipe operator (`|`), which composes any runnable into a pipeline.

It feels more sophisticated. It is not more agentic. Ask the key question: *who decided that translation happens after summarisation?* You did, when you wrote the `|`. The model has no say. Run this a thousand times and the path is identical every time.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text in exactly one sentence.\n\nTEXT:\n{text}"
)
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this sentence into Hindi. Output only the translation.\n\n{english}"
)

strip_think = RunnableLambda(clean)          # our helper, reused as a pipeline step

# prompt -> model -> string -> clean -> reshape -> prompt -> model -> string -> clean
chain = (
    summarize_prompt | model | StrOutputParser() | strip_think
    | (lambda summary: {"english": summary})
    | translate_prompt | model | StrOutputParser() | strip_think
)

ARTICLE = """
LangChain is an open-source framework for building applications on top of large language
models. It provides standard interfaces for models, tools, and memory, so that the same
application code can run against many different model providers. Its companion library,
LangGraph, adds durable execution, state and control flow for long-running agents.
"""

print(chain.invoke({"text": ARTICLE}))

### What to notice

- Two model calls, one hardcoded order. **The control flow lives in your source code**, visible on the page.
- This is a **workflow**, not an agent — and that is often exactly what you want. It is cheap, fast, debuggable and completely predictable.
- A very large fraction of production "AI features" are this, and should stay this.

Autonomy score: **still zero**. The model contributed intelligence, but no *decisions*.

---
## Level 2 — A router

### The model makes its first decision

Here the model finally gets a choice: read a support ticket and decide which department handles it. We use `with_structured_output`, which forces the model to answer in a schema we define (via Pydantic) rather than in free text — so we get a real Python object back, not a string we have to parse.

This is the first drop of agency. But notice the shape of it: the model chooses **one label**, once, and then *your* `if/elif` decides what actually happens.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Route(BaseModel):
    """Which department should handle this support ticket."""
    department: Literal["billing", "technical", "general"] = Field(
        description="The department best suited to handle the ticket"
    )
    reason: str = Field(description="One short sentence explaining the choice")

# method="json_schema" asks Groq to *guarantee* the reply matches the schema.
router = model.with_structured_output(Route, method="json_schema")

# --- your code owns what happens after the decision ---
def handle_billing(t):   return f"[BILLING BOT]   Opened refund case for: {t!r}"
def handle_technical(t): return f"[TECH BOT]      Created engineering ticket for: {t!r}"
def handle_general(t):   return f"[GENERAL BOT]   Sent FAQ article for: {t!r}"

def dispatch(ticket: str) -> str:
    # Give an explicit instruction -- a bare ticket invites a chatty reply instead of a decision.
    decision = router.invoke(f"Classify this support ticket into a department.\n\nTICKET: {ticket}")
    print(f"  model chose -> {decision.department:10} ({decision.reason})")
    if decision.department == "billing":
        return handle_billing(ticket)
    elif decision.department == "technical":
        return handle_technical(ticket)
    return handle_general(ticket)

for ticket in [
    "I was charged twice for my subscription this month.",
    "The app crashes every time I upload a PDF over 10 MB.",
    "What are your office hours?",
]:
    print(f"\nTicket: {ticket}")
    print(" ", dispatch(ticket))

### What to notice

- The model **influenced** the control flow for the first time — a real, if tiny, piece of agency.
- But the *set* of possible paths was fixed by you: three branches, no more. The model picked from a menu you wrote.
- It decides **once**. There is no "see the result, then decide again."

This pattern is called **routing** and it is one of the most valuable workflow patterns in production — it gives you the model's judgement while keeping full control of behaviour. Still a workflow, though.

Autonomy score: **1 decision, from a fixed menu, with no feedback**.

### Two details that will save you an afternoon

Both were learned the hard way while writing this notebook:

- **Always give an explicit instruction.** Passing the bare ticket text (`router.invoke(ticket)`) makes the model answer the *ticket* conversationally — "As an AI assistant, I'm available 24/7!" — instead of classifying it, and the call fails with `tool_use_failed`. Say what you want done: *"Classify this support ticket."*
- **`method="json_schema"`** uses Groq's constrained decoding, so the reply is *structurally guaranteed* to match `Route`. The default (`function_calling`) asks nicely and usually works. Prefer the guarantee when the output feeds an `if` statement.

---
## Level 3 — Tool calling

### The model chooses an *action* — but still cannot take it

Now we give the model **tools**. A tool is just a Python function plus a description the model can read. The `@tool` decorator turns the function signature and docstring into a JSON schema that gets sent to the model.

This is a genuine jump: instead of choosing from a menu of three labels, the model now chooses *which function to run* **and** *what arguments to pass it* — arguments it has to extract from natural language itself.

In [ ]:
from langchain.tools import tool

# A tiny fake "world" our tools can act on
_INVENTORY = {"laptop": 4, "monitor": 12, "keyboard": 0}
_PRICES    = {"laptop": 74999, "monitor": 15999, "keyboard": 2499}

def _normalize(item: str) -> str:
    """The model says 'laptops'; our catalogue says 'laptop'. Tools must be forgiving."""
    item = item.lower().strip()
    if item.endswith("s") and item[:-1] in _INVENTORY:
        item = item[:-1]
    return item

@tool
def check_stock(item: str) -> str:
    """Check how many units of an item are currently in stock."""
    item = _normalize(item)
    if item not in _INVENTORY:
        return f"Unknown item: {item}. Known items: {list(_INVENTORY)}"
    return f"{item}: {_INVENTORY[item]} units in stock"

@tool
def get_price(item: str) -> str:
    """Get the unit price of an item in Indian Rupees."""
    item = _normalize(item)
    if item not in _PRICES:
        return f"Unknown item: {item}"
    return f"{item}: Rs.{_PRICES[item]}"

@tool
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression such as '74999 * 3'."""
    if not set(expression) <= set("0123456789+-*/(). "):
        return "Error: only numbers and + - * / ( ) are allowed"
    return str(eval(expression))

TOOLS = [check_stock, get_price, calculate]

# Inspect what the model actually receives:
print("Tool name       :", check_stock.name)
print("Tool description:", check_stock.description)
print("Tool schema     :", check_stock.args)

Two things in that cell matter more than they look:

- **The docstring is not a comment — it is the model's only documentation.** It is sent to the model verbatim as the tool description. A vague docstring is a vague API, and the model will misuse the tool. Write them for the model, not for yourself.
- **`_normalize` exists because model input is messy.** Asked about "laptops", the model will happily pass `item="laptops"`, and a strict lookup fails. Real tools should be forgiving about plurals, case and whitespace — defensive parsing at the tool boundary prevents a large share of agent failures.

In [ ]:
model_with_tools = model.bind_tools(TOOLS)

response = model_with_tools.invoke("How many laptops do we have in stock?")

print("content   :", repr(clean(response)))          # empty -- it has nothing to SAY yet
print("tool_calls:", response.tool_calls)
print("finish    :", response.response_metadata["finish_reason"])

### What to notice — this is the crucial cell in the notebook

Look carefully at that output:

1. **`content` is empty.** The model produced no answer for the user. It cannot answer yet — it does not know the stock level.
2. **`tool_calls` is populated** with `name`, `args` and an `id`. The model worked out the argument from the English sentence entirely on its own — quite possibly as `"laptops"`, plural, which is exactly why `_normalize` exists.
3. **`finish_reason` is `'tool_calls'`,** not `'stop'`. The provider is telling you: *I did not finish, I am waiting for you.*

And now the part everyone misses:

> **Nothing ran.** The function `check_stock` was never called. The model *cannot* call it. An LLM emits text; that is all it can ever do. `tool_calls` is a **request**, not an action.

Prove it to yourself:

In [ ]:
print("Did the tool run? The model only produced a REQUEST:\n")
for tc in response.tool_calls:
    print(f"  name : {tc['name']}")
    print(f"  args : {tc['args']}")
    print(f"  id   : {tc['id']}")

print("\nWe have to run it ourselves:")
result = check_stock.invoke(response.tool_calls[0]["args"])
print("  result:", result)

print("\n>>> The model still has not seen this result. The conversation is stuck here.")

### The gap

We are one step away from an agent, and the missing step is glaring: **the model has never seen the tool's result.** We have a question, a request, and an answer sitting in a Python variable that the model knows nothing about.

To close the gap we must:
1. send the result *back* to the model, and
2. let it decide what to do next — answer, or call another tool.

Step 2 is the interesting one. If we let it decide *again*, and again, until it is satisfied... we have a **loop**. And that loop is the agent.

Autonomy score: **1 action chosen, arguments invented, no execution, no feedback**.

---
# ⭐ Level 4 — The agent loop

## This is the line. Everything above was a workflow; everything from here is an agent.

An agent is not a model, and it is not a prompt. **An agent is a loop.** Specifically:

```
   +---------------------------------------------------+
   |                                                    |
   v                                                    |
 [ MODEL ] --tool_calls?--> yes --> [ RUN TOOLS ] ------+
     |                                    ^
     |                                    | results appended to history
     no                                   |
     |                              (the model SEES what happened)
     v
 [ FINAL ANSWER ] -- loop ends
```

Three properties make this qualitatively different from everything above:

1. **The model observes the consequences of its own actions.** That is a feedback loop — the defining feature of agency.
2. **The number of steps is not known in advance.** It depends on what the tools return at runtime. Your code cannot predict the path.
3. **The model decides when to stop.** The loop's exit condition is `tool_calls == []` — a decision made by the model, not by you.

Let's write it by hand. It is about fifteen lines, and understanding these fifteen lines is worth more than any framework.

In [ ]:
from langchain.messages import HumanMessage, SystemMessage

TOOL_REGISTRY = {t.name: t for t in TOOLS}

def run_agent(user_request: str, max_steps: int = 6, verbose: bool = True):
    """A complete AI agent in ~15 lines. This is the whole idea."""

    messages = [
        SystemMessage("You are an inventory assistant. Use the tools to answer accurately. "
                      "Never guess a number you could look up."),
        HumanMessage(user_request),
    ]

    for step in range(1, max_steps + 1):
        ai_msg = model_with_tools.invoke(messages)     # 1. THINK: what should I do?
        messages.append(ai_msg)                        #    (keep the request in history!)

        if not ai_msg.tool_calls:                      # 2. DECIDE: am I done?
            if verbose:
                print(f"\n[step {step}] no tool calls -> FINISHED")
            return clean(ai_msg), messages

        for tc in ai_msg.tool_calls:                   # 3. ACT: run what it asked for
            if verbose:
                print(f"[step {step}] calling {tc['name']}({tc['args']})", end="")
            tool_msg = TOOL_REGISTRY[tc["name"]].invoke(tc)   # returns a ToolMessage
            if verbose:
                print(f"  ->  {tool_msg.content}")
            messages.append(tool_msg)                  # 4. OBSERVE: model will see this

    return "[stopped: hit max_steps]", messages


answer, history = run_agent(
    "What would it cost to buy 3 laptops and 2 monitors? Check they're in stock first."
)
print("\n" + "=" * 70)
print("FINAL ANSWER:\n")
show(answer)

### Read that output again

You asked one question. The agent decided, entirely on its own, to:

- call `check_stock` for laptops, then for monitors,
- call `get_price` for each,
- call `calculate` to multiply and add,
- and only then answer you.

**You never told it that sequence.** You never wrote "first check stock, then look up prices, then multiply." There is no `if` statement anywhere in `run_agent` that mentions stock or prices. The plan was constructed at runtime by the model, based on what the tools kept telling it.

That is the difference between a workflow and an agent, and it is not a matter of degree — it is a change in kind. In a workflow you can read the code and know the path. Here you cannot: **the trajectory is data, not code.**

In [ ]:
# The full trajectory. This message list IS the agent's state and its entire memory.
print(f"{len(history)} messages accumulated:\n")
for i, m in enumerate(history):
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        detail = "REQUESTS -> " + ", ".join(f"{tc['name']}({tc['args']})" for tc in m.tool_calls)
    elif kind == "ToolMessage":
        detail = f"RESULT   <- {m.content}"
    else:
        detail = (clean(m) or "")[:90].replace("\n", " ")
    print(f"{i:>2}. {kind:<14} {detail}")

### The anatomy of an agent

Every agent, in every framework, is these six parts. You just built all six:

| Component | In our code | Without it you get |
|---|---|---|
| **Model** | `model_with_tools` | Nothing — no decisions can be made |
| **Tools** | `TOOLS` | A chatbot that can't affect the world |
| **Loop** | `for step in range(...)` | Level 3: a request nobody fulfils |
| **Memory / state** | `messages` list | Amnesia — every step starts from scratch |
| **Goal / instructions** | `SystemMessage` + user request | Aimless behaviour |
| **Stop condition** | `if not ai_msg.tool_calls` | An infinite loop |

Note the last row. `max_steps` is *your* safety net; `if not tool_calls` is the *model's* exit. An agent that cannot decide it is finished is not usable, and an agent whose only stop is your step limit is broken.

---
## Level 4b — The same agent, prebuilt

Now that you know what the loop does, you never have to write it again. `create_agent` builds exactly the loop above on top of LangGraph, plus retries, streaming, persistence, interrupts and error handling.

**Use the prebuilt one in real projects.** Write the loop by hand once, for understanding — which you just did.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt=("You are an inventory assistant. Use the tools to answer accurately. "
                   "Never guess a number you could look up."),
)

result = agent.invoke({"messages": [HumanMessage("Is the keyboard in stock, and what does it cost?")]})

print("messages:", [type(m).__name__ for m in result["messages"]])
print()
show(result["messages"][-1])

### Watching it think, step by step

`.stream(..., stream_mode="updates")` emits each node of the loop as it completes — `model`, then `tools`, then `model` again. This is the single most useful debugging tool you have when an agent misbehaves, because it shows you *where* the trajectory went wrong.

In [ ]:
question = "We're buying 2 monitors. What's the total, and do we have enough stock?"

for update in agent.stream({"messages": [HumanMessage(question)]}, stream_mode="updates"):
    for node, payload in update.items():
        for m in payload.get("messages", []):
            if getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    print(f"[{node:<5}] THINK -> call {tc['name']}({tc['args']})")
            elif type(m).__name__ == "ToolMessage":
                print(f"[{node:<5}] OBSERVE <- {m.content}")
            else:
                print(f"[{node:<5}] ANSWER: {clean(m)[:200]}")

---
## Level 4c — Memory: making the agent stateful

Our hand-written loop forgot everything the moment it returned. Real agents run across many turns, so state has to survive.

LangGraph handles this with a **checkpointer**: it saves the message history after every step, keyed by a `thread_id`. A thread is one conversation. Same `thread_id` means the agent remembers; a different one means a clean slate.

`InMemorySaver` keeps this in RAM (perfect for a notebook). Swap in a Postgres or SQLite checkpointer in production and agents survive process restarts — the same API.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt="You are an inventory assistant. Be concise.",
    checkpointer=InMemorySaver(),            # <- the only line that adds memory
)

alice = {"configurable": {"thread_id": "alice-session"}}

r1 = memory_agent.invoke({"messages": [HumanMessage("How many monitors do we have?")]}, alice)
print("Turn 1:", clean(r1["messages"][-1])[:160])

# No repetition of "monitors" -- it must remember what we were talking about.
r2 = memory_agent.invoke({"messages": [HumanMessage("And what do they cost each?")]}, alice)
print("Turn 2:", clean(r2["messages"][-1])[:160])

r3 = memory_agent.invoke({"messages": [HumanMessage("So what's the total for all of them?")]}, alice)
print("Turn 3:", clean(r3["messages"][-1])[:200])

In [ ]:
# A different thread_id is a different conversation -- it has no idea what "they" means.
bob = {"configurable": {"thread_id": "bob-session"}}
r = memory_agent.invoke({"messages": [HumanMessage("And what do they cost each?")]}, bob)

print("Fresh thread:", clean(r["messages"][-1])[:250])
print("\nMessages on Alice's thread:", len(r3["messages"]))
print("Messages on Bob's thread   :", len(r["messages"]))

### What to notice

- The model itself is **stateless**. It is the message list, replayed on every call, that creates the illusion of memory. There is no hidden server-side conversation.
- This is why long agent runs get expensive: every step re-sends the entire history. Context management (summarisation, trimming) becomes a real engineering concern — LangChain ships a `SummarizationMiddleware` for exactly this.
- Memory is what makes an agent an ongoing *participant* rather than a one-shot *function*.

---
# Part 3 — Agentic design patterns

We now have an agent. **Agentic AI** is what you get when you start *composing* agency deliberately, rather than relying on one loop to figure everything out.

Four patterns cover most of the field (this taxonomy is Andrew Ng's, and it has held up well):

| Pattern | Idea | Status here |
|---|---|---|
| **Tool use** | The model acts on the world | ✅ built at Level 3–4 |
| **Reflection** | The model critiques and revises its own work | ↓ next |
| **Planning** | Decompose a goal into steps before acting | ↓ below |
| **Multi-agent collaboration** | Specialists delegate to each other | ↓ the finale |

The higher up this list you go, the more the word "agentic" is earned.

### Pattern 1 — Reflection

A single LLM pass produces first-draft quality. Reflection makes the model its own reviewer: **generate → critique → revise**. It is often the cheapest large quality win available, because the critique step is a genuinely different task from the generation step, and models are better at spotting flaws than avoiding them.

Notice there are no tools here at all. **Reflection is agentic without being tool-using** — the "action" is producing a critique, and the "observation" is reading it back.

In [ ]:
def reflect(task: str, rounds: int = 2) -> str:
    """generate -> critique -> revise, repeated."""
    draft = clean(model.invoke(f"Write a short Python function for this task.\n\nTASK: {task}"))
    print("--- DRAFT ---")
    print(draft[:600])

    for i in range(1, rounds + 1):
        critique = clean(model.invoke(
            f"You are a strict senior code reviewer.\n\nTASK: {task}\n\nCODE:\n{draft}\n\n"
            "List the concrete flaws: edge cases, error handling, naming, complexity. "
            "Be specific and terse. If the code is genuinely correct and complete, reply exactly: APPROVED"
        ))
        print(f"\n--- CRITIQUE {i} ---")
        print(critique[:600])

        if "APPROVED" in critique[:60].upper():
            print("\n>>> reviewer approved; stopping early")
            break

        draft = clean(model.invoke(
            f"TASK: {task}\n\nCURRENT CODE:\n{draft}\n\nREVIEW:\n{critique}\n\n"
            "Rewrite the code addressing every point. Output only the final code."
        ))
        print(f"\n--- REVISION {i} ---")
        print(draft[:600])

    return draft

final_code = reflect("Parse a duration string like '2h30m' or '45s' into total seconds.")

Note the `APPROVED` check: the loop can **exit early when the model judges the work good enough**. That is the same model-decides-when-to-stop property from Level 4, applied to quality instead of task completion.

### Pattern 2 — Planning

A plain agent loop is *reactive*: it decides one step at a time. Planning makes it **deliberate** — write the whole plan first, then execute it. This buys you an inspectable artifact (you can log the plan, show it to a user, or reject it before a single tool runs) and it keeps long tasks from drifting.

We use structured output again so the plan comes back as real Python objects.

In [ ]:
from typing import List

class Step(BaseModel):
    """A single step in a plan."""
    n: int = Field(description="Step number, starting at 1")
    action: str = Field(description="What to do, in one imperative sentence")
    tool: Literal["check_stock", "get_price", "calculate", "none"] = Field(
        description="Which tool this step needs, or 'none' for pure reasoning"
    )

class Plan(BaseModel):
    """An ordered plan for answering the user's request."""
    goal: str = Field(description="Restate the user's goal in one sentence")
    steps: List[Step] = Field(description="The ordered steps, at most 6")

planner = model.with_structured_output(Plan, method="json_schema")

GOAL = "Work out whether we can fulfil an order of 2 laptops and 3 monitors, and what it costs."
plan = planner.invoke(
    f"Make a plan to answer this using only these tools: check_stock, get_price, calculate.\n\n{GOAL}"
)

print("GOAL:", plan.goal, "\n")
for s in plan.steps:
    print(f"  {s.n}. [{s.tool:<12}] {s.action}")

In [ ]:
# Hand the approved plan to the agent as its instructions -- plan first, then execute.
plan_text = "\n".join(f"{s.n}. {s.action}" for s in plan.steps)

result = agent.invoke({"messages": [HumanMessage(
    f"{GOAL}\n\nFollow this plan exactly, using the tools:\n{plan_text}"
)]})

show(result["messages"][-1])

**Why bother?** Because the plan is now a *checkpoint you control*. In a real system you would show that numbered list to a human, or validate it against a policy, before letting any tool run. Reactive agents give you no such handle — by the time you see what happened, it has happened.

---
# ⭐ Part 4 — Multi-agent systems

## This is where "Agentic AI" stops being a synonym for "an agent"

One agent with twelve tools degrades badly: the prompt gets crowded, tool descriptions blur together, and the model starts picking wrong tools. The fix mirrors how human organisations scale — **specialisation plus delegation**.

The **supervisor pattern** is the most common shape:

```
                        [ SUPERVISOR ]
                     decomposes & delegates
                    /                      \
                   v                        v
          [ RESEARCH AGENT ]         [ MATH AGENT ]
          own model + own tools      own model + own tools
          own message history        own message history
```

The key trick: **an agent can be wrapped in `@tool` and handed to another agent.** From the supervisor's point of view, a whole specialist agent is just a function it can call. Agents compose.

In [ ]:
# --- Specialist 1: inventory research ---
research_agent = create_agent(
    model=model,
    tools=[check_stock, get_price],
    system_prompt="You look up inventory facts. Report exactly what the tools return. Never do arithmetic.",
)

# --- Specialist 2: arithmetic ---
math_agent = create_agent(
    model=model,
    tools=[calculate],
    system_prompt="You do arithmetic using the calculator tool. Never guess a number; always compute it.",
)

# --- Wrap each agent as a tool the supervisor can call ---
@tool
def inventory_specialist(question: str) -> str:
    """Ask the inventory specialist about stock levels or prices. Ask one clear question."""
    out = research_agent.invoke({"messages": [HumanMessage(question)]})
    return clean(out["messages"][-1])

@tool
def math_specialist(question: str) -> str:
    """Ask the math specialist to compute something. State the numbers explicitly."""
    out = math_agent.invoke({"messages": [HumanMessage(question)]})
    return clean(out["messages"][-1])

supervisor = create_agent(
    model=model,
    tools=[inventory_specialist, math_specialist],
    system_prompt=(
        "You are a supervisor coordinating two specialists. "
        "Break the user's request into steps and delegate each step to the right specialist. "
        "Never look anything up or calculate anything yourself. "
        "When you have all the pieces, give the user a single clear answer."
    ),
)

print("Supervisor ready with sub-agents:", [t.name for t in [inventory_specialist, math_specialist]])

In [ ]:
task = "A customer wants 3 laptops and 2 monitors. Can we fulfil it, and what is the total bill?"

for update in supervisor.stream({"messages": [HumanMessage(task)]}, stream_mode="updates"):
    for node, payload in update.items():
        for m in payload.get("messages", []):
            if getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    print(f"\nSUPERVISOR delegates to {tc['name']}:")
                    print(f"    \"{tc['args']['question']}\"")
            elif type(m).__name__ == "ToolMessage":
                print(f"  {m.name} replies: {m.content[:220]}")
            else:
                print("\n" + "=" * 70 + "\nFINAL:\n")
                show(m)

### What just happened — and why it is a different category

Watch the delegation trace. The supervisor **invented sub-questions that nobody wrote**: it turned one customer request into a stock question, a price question, and an arithmetic question, then assembled the answers.

That is the qualitative jump:

| | Single agent (L4) | Multi-agent system (L5) |
|---|---|---|
| Decides | which *tool* to call | which *agent* to consult, and **what to ask it** |
| Generates | tool arguments | **new sub-goals** in natural language |
| Context | one shared history | isolated per agent — each specialist sees only its own job |
| Failure | wrong tool call | one agent's bad answer silently poisons another's input |
| Cost | N model calls | N × M — every delegation is a full agent run |

The context isolation is the real engineering benefit: the math agent never sees inventory chatter, so its prompt stays small and its tool choice stays sharp.

### When *not* to do this

Multi-agent is genuinely expensive and genuinely harder to debug. Reach for it only when:

- ✅ one agent's tool list has grown past ~10 tools and it is picking wrong ones,
- ✅ sub-tasks need genuinely different instructions, models, or permissions,
- ✅ you want isolated context per specialist.

Do **not** reach for it because it sounds impressive. A well-prompted single agent beats a badly-coordinated committee almost every time, and errors compound across handoffs: three agents at 90% reliability each give you ~73% end-to-end.

---
# Part 5 — Control and safety

The moment you cross the L3/L4 line, you have given up knowing what your program will do. That is the *point* of agents, and it is also the whole risk. Autonomy without limits is not sophistication, it is an incident waiting to happen.

Three controls, in order of importance:

1. **Budget limits** — an agent that loops forever burns real money.
2. **Human approval** — some actions must never fire unreviewed.
3. **Scoped tools** — the safest guardrail is a tool that *cannot* do the dangerous thing.

LangChain implements the first two as **middleware** you attach to `create_agent`.

### Limiting the loop

`max_steps` in our hand-written loop was crude. `ToolCallLimitMiddleware` caps tool calls per run or per thread, and `exit_behavior="end"` stops the agent cleanly instead of raising.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

@tool
def roll_dice(sides: int) -> str:
    """Roll a die with the given number of sides."""
    return "4"

limited_agent = create_agent(
    model=model,
    tools=[roll_dice],
    system_prompt="Roll dice when asked, one roll per tool call.",
    middleware=[ToolCallLimitMiddleware(thread_limit=3, exit_behavior="end")],
)

out = limited_agent.invoke({"messages": [HumanMessage("Roll a 6-sided die 10 separate times.")]})

tool_runs = sum(1 for m in out["messages"] if type(m).__name__ == "ToolMessage")
print(f"Asked for 10 rolls, actually executed: {tool_runs}")
print("Agent stopped with:", clean(out["messages"][-1])[:160])

The agent wanted ten rolls and was cut off after a handful. Note the exact number: the limit trips on the call that *exceeds* it, so you will see one more execution than the limit — the middleware detects `4/3` and ends the run. Set the limit as a budget ceiling, not an exact count.

Also note *how* it stopped: `exit_behavior="end"` terminates the loop cleanly and leaves an explanatory message in the history, rather than raising an exception you would have to catch.

### Human-in-the-loop: pausing before a dangerous action

This is the most important safety primitive in the whole notebook. `HumanInTheLoopMiddleware` **interrupts execution** before a named tool runs, and hands control back to you. The agent's full state is saved to the checkpointer (which is why HITL *requires* one), so the pause can last milliseconds or days.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def delete_file(path: str) -> str:
    """Permanently delete a file at the given path."""
    return f"DELETED {path}"       # pretend this is destructive and irreversible

safe_agent = create_agent(
    model=model,
    tools=[delete_file],
    system_prompt="You manage files for the user.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"delete_file": True})],
    checkpointer=InMemorySaver(),           # required: the pause must be persisted
)

session = {"configurable": {"thread_id": "review-1"}}
paused = safe_agent.invoke(
    {"messages": [HumanMessage("Delete the file reports/q3_draft.csv")]}, session
)

print("Did the agent pause? ->", "__interrupt__" in paused)
request = paused["__interrupt__"][0].value["action_requests"][0]
print("\nAWAITING APPROVAL:")
print("   tool :", request["name"])
print("   args :", request["args"])
print("\nNothing has been deleted. Execution is frozen mid-loop.")

In [ ]:
# --- The human approves: resume and let the tool run ---
approved = safe_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), session)

print("APPROVED PATH")
print("  messages:", [type(m).__name__ for m in approved["messages"]])
print("  result  :", clean(approved["messages"][-1])[:180])

In [ ]:
# --- The human rejects: the tool never runs, and the agent is told why ---
session2 = {"configurable": {"thread_id": "review-2"}}
safe_agent.invoke({"messages": [HumanMessage("Delete the file reports/q4_forecast.csv")]}, session2)

rejected = safe_agent.invoke(
    Command(resume={"decisions": [
        {"type": "reject", "message": "Denied by reviewer: that file is still in use."}
    ]}),
    session2,
)

print("REJECTED PATH")
for m in rejected["messages"]:
    print(f"  {type(m).__name__:<12} {clean(m)[:120] or m.content[:120]}")

### What to notice

- The rejection is not an exception — it is fed back to the model **as a `ToolMessage`**, so the agent understands what happened and explains it to the user. The loop absorbs the denial gracefully.
- Available decisions are `approve`, `edit` (change the arguments before running), `reject`, and `respond` (reply to the model without running the tool).
- `interrupt_on={"delete_file": True}` — tools not listed run freely. Gate the destructive ones, let reads through.

### Other guardrails worth knowing

`langchain.agents.middleware` also ships: `ModelCallLimitMiddleware` (cap total model calls), `SummarizationMiddleware` (compress history when context fills), `PIIMiddleware` (detect/redact personal data), `ModelFallbackMiddleware` (switch models on failure), and `ToolRetryMiddleware`.

And the guardrail that beats all of them: **give the agent a tool that can only do the safe thing.** A `refund(order_id)` tool that caps at ₹5000 internally cannot be talked into refunding ₹5,00,000, no matter how clever the prompt injection.

---
# Part 6 — So which should you build?

The single most valuable skill here is **not** building agents. It is recognising when you do not need one.

```
Do you know the exact sequence of steps in advance?
    |
    YES --> Write a CHAIN. Done. (Level 1)
    |       Cheap, fast, testable, debuggable. Most production AI is this.
    |
    NO
    |
Is it a fixed set of known branches?
    |
    YES --> Write a ROUTER. (Level 2)
    |       Model's judgement, your control flow.
    |
    NO
    |
Does the next step depend on what previous steps returned?
    |
    YES --> You need an AGENT. (Level 4)
    |
    NO  --> You probably still want a workflow. Look again.
             |
Is the tool list large, or do sub-tasks need different rules/permissions?
    |
    YES --> MULTI-AGENT. (Level 5) Accept the cost and the debugging burden.
    |
    NO  --> One agent. Keep it that way as long as you can.
```

### The trade-off, stated plainly

| | Workflow | Agent |
|---|---|---|
| Predictable | ✅ same path every time | ❌ trajectory varies per run |
| Cost | ✅ known in advance | ❌ unbounded without limits |
| Latency | ✅ fixed | ❌ depends on step count |
| Debuggable | ✅ read the code | ❌ read the trace, per run |
| Testable | ✅ deterministic assertions | ❌ needs eval sets, not unit tests |
| Handles the unexpected | ❌ breaks | ✅ adapts |
| Open-ended tasks | ❌ impossible | ✅ the entire point |

You are trading **predictability for adaptability**, and you pay in cost, latency and debuggability. Make that trade deliberately, and only when the task actually requires it.

---
# Part 7 — Answering the original question

Now that you have built every level, the terminology is no longer abstract.

### AI Agent
**What you built at Level 4.** A single system with a model, tools, a loop, memory, and a stop condition — that decides its own next action at runtime. Countable. Deployable. One thing.

> *"The inventory agent handles stock queries."*

### Agentic AI
**The property, and the paradigm — visible from Level 4 upward, dominant at Level 5.** The design approach where you specify *goals* and *constraints* instead of *steps*, and where systems plan, delegate, self-correct and recover. Not a thing you deploy; a way you build.

> *"We rebuilt the ordering pipeline around agentic AI."*

### The sentence to remember

> **You build *AI agents*. You practise *Agentic AI*. The agent is the noun; agentic is how much of the control flow you handed to the model.**

### And the question that dissolves the debate

Whenever someone asks whether X "is really agentic", ignore the label and ask:

1. Does the model choose **what happens next**, or does my code? (Level 2 vs 4)
2. Does it see the **results of its own actions** and adapt? (the feedback loop)
3. Does it decide **when it is finished**? (the stop condition)
4. Can it **delegate** to other agents? (single vs. multi-agent)

Count the yeses. That number is more informative than any label — and unlike the label, it is something you can point at in the code.

### Glossary

| Term | Meaning |
|---|---|
| **Agent** | Model + tools + loop + memory that directs its own control flow |
| **Agentic** | Adjective: possessing runtime decision-making autonomy |
| **Agentic AI** | The paradigm of building with that autonomy; often implies multi-agent |
| **Agentic workflow** | Middle ground: LLM decisions inside partly-fixed control flow |
| **Workflow / chain** | Fixed, developer-defined sequence of LLM calls |
| **Tool** | A function the model can request, described by a JSON schema |
| **`tool_calls`** | The model's *request* to run a tool — a request, never an execution |
| **ReAct** | Reason + Act: the think→act→observe loop you wrote at Level 4 |
| **Trajectory** | The actual sequence of steps one agent run took |
| **Reflection** | Agent critiques and revises its own output |
| **Supervisor / orchestrator** | An agent whose tools are other agents |
| **Checkpointer** | Persistence layer storing agent state per `thread_id` |
| **Thread** | One conversation with its own saved state |
| **Middleware** | Hooks around the agent loop: limits, approvals, redaction, fallbacks |
| **Human-in-the-loop (HITL)** | Pausing the loop for human approval before an action |

---
## Exercises

Work through these before moving on — building beats reading.

1. **Break the loop.** Delete `messages.append(ai_msg)` from `run_agent` and run it. Explain the resulting behaviour in one sentence. (This is the single most common agent bug.)
2. **Add a tool.** Write `apply_discount(amount: float, percent: float)` and add it to `TOOLS`. Ask a question needing four chained tool calls. Did the agent find the path unaided?
3. **Force a failure.** Make `check_stock` return `"ERROR: inventory service unavailable"`. Does the agent retry, apologise, or hallucinate a number? Try improving the system prompt so it degrades honestly.
4. **Downgrade a level.** Rewrite the multi-agent supervisor as a plain chain. What did you gain? What became impossible?
5. **Gate a tool.** Add `HumanInTheLoopMiddleware` to the inventory agent so `calculate` needs approval, then use `{"type": "edit"}` to change the arguments before it runs.
6. **Count the yeses.** Take the four diagnostic questions from Part 7 and score every level in this notebook, 0–4.

## Where to go next in this repo

- [langchain/1-langchainintro.ipynb](langchain/1-langchainintro.ipynb) — LangChain fundamentals
- [langchain/2-modelintegration.ipynb](langchain/2-modelintegration.ipynb) — swapping providers
- [langchain/3-tools.ipynb](langchain/3-tools.ipynb) — tools in depth
- [langchain/4-messages.ipynb](langchain/4-messages.ipynb) — the message types that carry all of this

*(Paths assume this notebook sits in the repo root. If you move it into `langchain/`, drop the prefix.)*

Beyond that: LangGraph's graph API for custom agent topologies (cycles, branches, parallel agents), evaluation with LangSmith, and streaming tokens to a UI.